In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
PROJECT_PATH = "/content/drive/MyDrive/Telco-customer-churn"

In [3]:
import pandas as pd
import sqlite3
import numpy as np

In [4]:
df = pd.read_csv(
    f"{PROJECT_PATH}/Data/WA_Fn-UseC_-Telco-Customer-Churn.csv"
)

df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [5]:
df.dtypes

,0
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [7]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df.dropna(subset=['TotalCharges'], inplace=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7032 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7032 non-null   object 
 1   gender            7032 non-null   object 
 2   SeniorCitizen     7032 non-null   int64  
 3   Partner           7032 non-null   object 
 4   Dependents        7032 non-null   object 
 5   tenure            7032 non-null   int64  
 6   PhoneService      7032 non-null   object 
 7   MultipleLines     7032 non-null   object 
 8   InternetService   7032 non-null   object 
 9   OnlineSecurity    7032 non-null   object 
 10  OnlineBackup      7032 non-null   object 
 11  DeviceProtection  7032 non-null   object 
 12  TechSupport       7032 non-null   object 
 13  StreamingTV       7032 non-null   object 
 14  StreamingMovies   7032 non-null   object 
 15  Contract          7032 non-null   object 
 16  PaperlessBilling  7032 non-null   object 
 17  

In [8]:
df['Churn_Flag'] = df['Churn'].map({'Yes': 1, 'No': 0})
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,Churn_Flag
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.50,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


In [9]:
df['Revenue_at_risk'] = df.apply(lambda x: x['MonthlyCharges'] if x['Churn_Flag'] == 1 else 0, axis = 1)

In [10]:
df.to_csv('/content/WA_Fn-UseC_-Telco-Customer-Churn-Processed.csv', index=False)

In [11]:
# Load into SQLite so we can write real SQL queries
import sqlite3

conn = sqlite3.connect('../content/telco_churn.db')
df.to_sql('customers', conn, if_exists='replace', index=False)

print(f"Rows loaded: {pd.read_sql('SELECT COUNT(*) as n FROM customers', conn).iloc[0,0]}")
print(f"Churned customers: {df['Churn_Flag'].sum()}")
print(f"Total monthly revenue at risk: ₹{df['Revenue_at_risk'].sum():,.0f}")

Rows loaded: 7032
Churned customers: 1869
Total monthly revenue at risk: ₹139,131


In [15]:
def run_query(label, query):
  print(f"\n{'='*100}")
  print(f" {label}")
  print('='*100)
  result=pd.read_sql(query, conn)
  print(result.to_string(index=False))
  print('='*100)
  return result

In [16]:
#Headline Churn rate
q1=run_query("Overall Churn Rate", """
  select count(*) as total_customers,
  sum(Churn_Flag) as churned,
  round(100.0*sum(Churn_Flag)/count(*),1) as churn_rate_pct,
  round(sum(Revenue_at_risk),0) as monthly_rev_at_risk
  from customers
""")


 Overall Churn Rate
 total_customers  churned  churn_rate_pct  monthly_rev_at_risk
            7032     1869            26.6             139131.0


### 💡 Insight: Overall churn is putting significant recurring revenue at risk

**What the data shows:** Out of 7032 customers, 1869 have churned, resulting in an overall churn rate of 26.6%. This represents approximately $139131 in monthly recurring revenue at risk.

**Why it matters:** More than one in four customers leave the business, creating a substantial recurring revenue loss that directly impacts long-term growth and customer lifetime value.

**Action:** Make churn reduction a company-wide KPI by prioritizing high-risk customer segments and tracking monthly revenue at risk alongside churn.


In [17]:
q2=run_query("Churn by Contract Type", """
  select Contract, count(*) as total_customers,
  sum(Churn_Flag) as churned,
  round(100.0*sum(Churn_Flag)/count(*),1) as churn_rate_pct,
  round(sum(Revenue_at_risk),0) as monthly_rev_at_risk
  from customers
  group by Contract
  order by churn_rate_pct desc
""")


 Churn by Contract Type
      Contract  total_customers  churned  churn_rate_pct  monthly_rev_at_risk
Month-to-month             3875     1655            42.7             120847.0
      One year             1472      166            11.3              14118.0
      Two year             1685       48             2.8               4165.0


### 💡 Insight: Contract Type drives churn more than any other factor

**What the data shows:** Month-to-month customers churn at 43% vs 11% on
one-year and 3% on two-year contracts.

**Why it matters:** The majority of the customer base is on month-to-month —
meaning the company's revenue base is structurally fragile.

**Action:** Priority retention spend should target month-to-month customers
in their first 12 months with contract upgrade incentives.


In [19]:
q3=run_query("Churn Rate by Tenure Segment", """
  select case when tenure <= 12 then '1-12 months'
              when tenure <= 24 then '13-24 months'
              when tenure <= 48 then '25-48 months'
              else '48+ months' end as tenure_segment,
  count(*) as total_customers,
  sum(Churn_Flag) as churned,
  round(100.0*sum(Churn_Flag)/count(*),1) as churn_rate_pct,
  round(avg(MonthlyCharges),0) as avg_monthly_charges
  from customers
  group by tenure_segment
  order by churn_rate_pct desc
""")


 Churn Rate by Tenure Segment
tenure_segment  total_customers  churned  churn_rate_pct  avg_monthly_charges
   1-12 months             2175     1037            47.7                 56.0
  13-24 months             1024      294            28.7                 61.0
  25-48 months             1594      325            20.4                 66.0
    48+ months             2239      213             9.5                 74.0


### 💡 Insight: Most customers who churn do so within their first year

**What the data shows:** Customers with 1–12 months tenure have the highest churn rate at 47.7%, which steadily declines to 9.5% for customers staying longer than 48 months.

**Why it matters:** Customer loyalty strengthens over time, making the first year the most critical period for retention. Preventing early churn can significantly improve customer lifetime value.

**Action:** Strengthen onboarding and customer engagement during the first 12 months through proactive support, personalized offers, and early retention campaigns.

In [20]:
q4=run_query("Revenue Profile: Churned vs Retained", """
  select Churn,
  count(*) as total_customers,
  round(avg(MonthlyCharges),0) as avg_monthly_charges,
  round(avg(TotalCharges),0) as avg_total_charges,
  round(sum(MonthlyCharges),0) as total_monthly_revenue
  from customers
  group by Churn
""")


 Revenue Profile: Churned vs Retained
Churn  total_customers  avg_monthly_charges  avg_total_charges  total_monthly_revenue
   No             5163                 61.0             2555.0               316530.0
  Yes             1869                 74.0             1532.0               139131.0


### 💡 Insight: Churned customers generate higher monthly revenue than retained customers

**What the data shows:** Churned customers pay an average of $74/month, compared to $61/month for retained customers, despite having lower average lifetime spending due to shorter tenure.

**Why it matters:** The company is disproportionately losing higher-value customers, increasing the financial impact of churn beyond customer count alone.

**Action:** Identify high-paying customers showing early churn signals and prioritize them with loyalty rewards, service improvements, or personalized retention offers.

In [21]:
q5 = run_query("Churn by Internet Service", """
SELECT
    InternetService,
    COUNT(*)                                          AS total,
    SUM(Churn_Flag)                                   AS churned,
    ROUND(100.0 * SUM(Churn_Flag) / COUNT(*), 1)     AS churn_rate_pct,
    ROUND(SUM(Revenue_At_Risk), 0)                    AS monthly_rev_at_risk
FROM customers
GROUP BY InternetService
ORDER BY churn_rate_pct DESC
""")


 Churn by Internet Service
InternetService  total  churned  churn_rate_pct  monthly_rev_at_risk
    Fiber optic   3096     1297            41.9             114300.0
            DSL   2416      459            19.0              22529.0
             No   1520      113             7.4               2302.0


### 💡 Insight: Fiber optic customers contribute the majority of churn-related revenue loss

**What the data shows:** Fiber optic customers churn at 41.9%, more than double DSL customers (19%) and nearly six times customers without internet (7.4%). This segment alone accounts for $114,300 in monthly revenue at risk.

**Why it matters:** Although Fiber optic customers are high-value, they are also the most likely to leave, making them the biggest contributor to revenue leakage.

**Action:** Investigate Fiber optic customer pain points such as pricing, service quality, or competition, and prioritize targeted retention initiatives for this segment.

In [22]:
q6 = run_query("Top High-Risk Segments", """
SELECT
    Contract,
    InternetService,
    PaymentMethod,
    COUNT(*)                                          AS total,
    SUM(Churn_Flag)                                   AS churned,
    ROUND(100.0 * SUM(Churn_Flag) / COUNT(*), 1)     AS churn_rate_pct,
    ROUND(AVG(MonthlyCharges), 0)                     AS avg_monthly_rev,
    ROUND(SUM(Revenue_At_Risk), 0)                    AS segment_rev_at_risk
FROM customers
GROUP BY Contract, InternetService, PaymentMethod
HAVING total > 30
ORDER BY churn_rate_pct DESC
LIMIT 10
""")


 Top High-Risk Segments
      Contract InternetService             PaymentMethod  total  churned  churn_rate_pct  avg_monthly_rev  segment_rev_at_risk
Month-to-month     Fiber optic          Electronic check   1307      789            60.4             87.0              68282.0
Month-to-month     Fiber optic              Mailed check    201      102            50.7             83.0               8407.0
Month-to-month     Fiber optic Bank transfer (automatic)    327      149            45.6             88.0              13062.0
Month-to-month     Fiber optic   Credit card (automatic)    293      122            41.6             88.0              10731.0
Month-to-month             DSL          Electronic check    474      192            40.5             49.0               8776.0
Month-to-month             DSL              Mailed check    367      113            30.8             49.0               5292.0
Month-to-month             DSL   Credit card (automatic)    185       50            27

### 💡 Insight: Month-to-month Fiber optic customers paying by Electronic Check are the highest-risk segment

**What the data shows:** Customers with Month-to-month contracts, Fiber optic service, and Electronic Check payments have a 60.4% churn rate—the highest among all customer segments—and account for $68,282 in monthly revenue at risk.

**Why it matters:** A relatively small customer segment contributes a disproportionately large share of churn-related revenue loss, making it the most valuable retention opportunity.

**Action:** Prioritize this segment with targeted offers such as automatic payment enrollment, contract upgrade discounts, and proactive customer outreach to reduce churn.

In [23]:
q7 = run_query("Monthly Revenue at Risk", """
SELECT
    ROUND(SUM(MonthlyCharges), 0)                              AS total_mrr,
    ROUND(SUM(Revenue_At_Risk), 0)                             AS revenue_at_risk,
    ROUND(100.0 * SUM(Revenue_At_Risk) / SUM(MonthlyCharges), 1) AS pct_mrr_at_risk
FROM customers
""")


 Monthly Revenue at Risk
 total_mrr  revenue_at_risk  pct_mrr_at_risk
  455661.0         139131.0             30.5


### 💡 Insight: Nearly one-third of monthly recurring revenue is at risk

**What the data shows:** The business generates $455,661 in monthly recurring revenue (MRR), of which $139,131—or 30.5%—comes from customers who eventually churn.

**Why it matters:** Losing nearly one-third of recurring revenue significantly impacts predictable cash flow, profitability, and long-term business growth. Even modest improvements in retention could recover substantial recurring revenue.

**Action:** Track Revenue at Risk as a core business KPI and prioritize retention efforts for high-value, high-risk customer segments to protect recurring revenue.

In [24]:
q8 = run_query("Retention Rate by Tenure Cohort", """
SELECT
    CASE
        WHEN tenure <= 6  THEN '0–6 months'
        WHEN tenure <= 12 THEN '7–12 months'
        WHEN tenure <= 24 THEN '13–24 months'
        WHEN tenure <= 36 THEN '25–36 months'
        ELSE '36+ months'
    END                                               AS cohort,
    COUNT(*)                                          AS total,
    SUM(Churn_Flag)                                   AS churned,
    ROUND(100.0 * (COUNT(*) - SUM(Churn_Flag)) / COUNT(*), 1) AS retention_rate_pct,
    ROUND(AVG(MonthlyCharges), 0)                     AS avg_monthly_rev
FROM customers
GROUP BY cohort
ORDER BY MIN(tenure)
""")


 Retention Rate by Tenure Cohort
      cohort  total  churned  retention_rate_pct  avg_monthly_rev
  0–6 months   1470      784                46.7             55.0
 7–12 months    705      253                64.1             59.0
13–24 months   1024      294                71.3             61.0
25–36 months    832      180                78.4             66.0
  36+ months   3001      358                88.1             72.0


In [33]:
early_churned = 784
recovery_pct = 0.10
avg_rev = 55
print(f"10% retention improvement in 0–6 cohort = ₹{early_churned * recovery_pct * avg_rev:,.0f}/month recovered")

10% retention improvement in 0–6 cohort = ₹4,312/month recovered


## 💡 Insight: The retention cliff is months 0–6, not 0–12

**What the data shows:** Customers in months 0–6 retain at only 46.7%.
Those who reach month 7 retain at 64.1% — a 17-point jump in 6 months.
By month 36+, retention reaches 88.1%.

**Why it matters:** The first 6 months is the highest-risk, highest-leverage
window in the entire customer lifecycle. Losing a customer here also means
losing a high-CLV customer — month 36+ customers pay ₹72/month vs ₹55
for new customers.

**Action:** Invest disproportionately in onboarding, early check-ins, and contract upgrade offers between months 1–6. A 10% reduction in churn within the 0–6 month cohort would recover approximately ₹4,312 in monthly recurring revenue (MRR), or about ₹51,700 annually.

In [37]:
import os

os.makedirs("sql", exist_ok=True)

In [39]:
# Auto-export all queries to sql/01_churn_analysis.sql
queries_to_export = {
    "Overall Churn Rate": """
  select count(*) as total_customers,
  sum(Churn_Flag) as churned,
  round(100.0*sum(Churn_Flag)/count(*),1) as churn_rate_pct,
  round(sum(Revenue_at_risk),0) as monthly_rev_at_risk
  from customers
""",

    "Churn by Contract Type": """
  select Contract, count(*) as total_customers,
  sum(Churn_Flag) as churned,
  round(100.0*sum(Churn_Flag)/count(*),1) as churn_rate_pct,
  round(sum(Revenue_at_risk),0) as monthly_rev_at_risk
  from customers
  group by Contract
  order by churn_rate_pct desc""",

    "Churn Rate by Tenure Segment": """
  select case when tenure <= 12 then '1-12 months'
              when tenure <= 24 then '13-24 months'
              when tenure <= 48 then '25-48 months'
              else '48+ months' end as tenure_segment,
  count(*) as total_customers,
  sum(Churn_Flag) as churned,
  round(100.0*sum(Churn_Flag)/count(*),1) as churn_rate_pct,
  round(avg(MonthlyCharges),0) as avg_monthly_charges
  from customers
  group by tenure_segment
  order by churn_rate_pct desc
""",

    "Revenue Profile: Churned vs Retained": """
  select Churn,
  count(*) as total_customers,
  round(avg(MonthlyCharges),0) as avg_monthly_charges,
  round(avg(TotalCharges),0) as avg_total_charges,
  round(sum(MonthlyCharges),0) as total_monthly_revenue
  from customers
  group by Churn
""",

    "Churn by Internet Service": """
SELECT
    InternetService,
    COUNT(*)                                          AS total,
    SUM(Churn_Flag)                                   AS churned,
    ROUND(100.0 * SUM(Churn_Flag) / COUNT(*), 1)     AS churn_rate_pct,
    ROUND(SUM(Revenue_At_Risk), 0)                    AS monthly_rev_at_risk
FROM customers
GROUP BY InternetService
ORDER BY churn_rate_pct DESC
""",

    "Top High-Risk Segments": """
SELECT
    Contract,
    InternetService,
    PaymentMethod,
    COUNT(*)                                          AS total,
    SUM(Churn_Flag)                                   AS churned,
    ROUND(100.0 * SUM(Churn_Flag) / COUNT(*), 1)     AS churn_rate_pct,
    ROUND(AVG(MonthlyCharges), 0)                     AS avg_monthly_rev,
    ROUND(SUM(Revenue_At_Risk), 0)                    AS segment_rev_at_risk
FROM customers
GROUP BY Contract, InternetService, PaymentMethod
HAVING total > 30
ORDER BY churn_rate_pct DESC
LIMIT 10
""",

    "Monthly Revenue at Risk": """
SELECT
    ROUND(SUM(MonthlyCharges), 0)                              AS total_mrr,
    ROUND(SUM(Revenue_At_Risk), 0)                             AS revenue_at_risk,
    ROUND(100.0 * SUM(Revenue_At_Risk) / SUM(MonthlyCharges), 1) AS pct_mrr_at_risk
FROM customers
""",

    "Retention Rate by Tenure Cohort": """
SELECT
    CASE
        WHEN tenure <= 6  THEN '0–6 months'
        WHEN tenure <= 12 THEN '7–12 months'
        WHEN tenure <= 24 THEN '13–24 months'
        WHEN tenure <= 36 THEN '25–36 months'
        ELSE '36+ months'
    END                                               AS cohort,
    COUNT(*)                                          AS total,
    SUM(Churn_Flag)                                   AS churned,
    ROUND(100.0 * (COUNT(*) - SUM(Churn_Flag)) / COUNT(*), 1) AS retention_rate_pct,
    ROUND(AVG(MonthlyCharges), 0)                     AS avg_monthly_rev
FROM customers
GROUP BY cohort
ORDER BY MIN(tenure)
"""
}

with open("sql/01_churn_analysis.sql", "w") as f:
    for label, query in queries_to_export.items():
        f.write(f"-- {label}\n{query.strip()}\n\n")

print("SQL file exported.")

SQL file exported.
